# ALLUVO Content‑Based Recommendation Notebook

Workflow

1. Load datasets  
2. Data cleaning  
3. Feature engineering  
4. Interaction score (watch + like + purchase)  
5. Reel feature matrix  
6. Build user profile  
7. Cosine similarity recommendation  
8. Generate Top‑K recommendations


In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

## Load Data

In [5]:
users = pd.read_csv("../data/raw/users.csv")
brands = pd.read_csv("../data/raw/brands.csv")
reels = pd.read_csv("../data/raw/reels.csv")
interactions = pd.read_csv("../data/raw/interactions.csv")

print(users.shape)
print(reels.shape)
print(interactions.shape)


(2000, 6)
(3000, 6)
(65298, 8)


## Data Cleaning

In [6]:
reels = reels.drop_duplicates(subset="reel_id")
interactions = interactions.dropna()

interactions.head()

,user_id,reel_id,view,watch_ratio,like,comment,purchase,timestamp
0,1,1191,1,0.84,0,0,0,2025-12-12 15:49:21.170953
1,1,2506,1,0.71,0,1,0,2025-12-28 15:49:21.544868
2,1,1764,1,0.67,0,0,0,2026-02-18 15:49:21.340695
3,1,2777,1,0.91,1,0,0,2026-02-17 15:49:21.619322
4,1,1074,1,0.70,0,0,0,2026-02-09 15:49:21.135591


## Interaction Score

In [7]:
interactions["interaction_score"] = (
    interactions["watch_ratio"] * 0.4 +
    interactions["like"] * 0.2 +
    interactions["purchase"] * 0.4
)

interactions[["watch_ratio","like","purchase","interaction_score"]].head()


,watch_ratio,like,purchase,interaction_score
0,0.84,0,0,0.336
1,0.71,0,0,0.284
2,0.67,0,0,0.268
3,0.91,1,0,0.564
4,0.70,0,0,0.280


## Feature Engineering for Reels

In [10]:
# 1. One-Hot Encoding for Category and Brand ID
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_cols = ["category", "brand_id"]
encoded_cats = encoder.fit_transform(reels[categorical_cols])

# تحويل الناتج إلى DataFrame لتسهيل الدمج
encoded_df = pd.DataFrame(
    encoded_cats,
    columns=encoder.get_feature_names_out(categorical_cols)
)

# 2. Min-Max Scaling for Price
scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(reels[["price"]])
price_df = pd.DataFrame(price_scaled, columns=["price_scaled"])

# 3. Combine Features & Build Matrix
reel_features = pd.concat(
    [reels[["reel_id"]], encoded_df, price_df],
    axis=1
)

# تعيين reel_id ليكون الـ Index لتسهيل البحث لاحقاً
reel_matrix = reel_features.set_index("reel_id")

reel_matrix.head()

,category_Beauty,category_Fashion,category_Fitness,category_Gaming,category_Home,category_Sports,category_Technology,brand_id_1,brand_id_2,brand_id_3,...,brand_id_22,brand_id_23,brand_id_24,brand_id_25,brand_id_26,brand_id_27,brand_id_28,brand_id_29,brand_id_30,price_scaled
reel_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.594909
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.149897
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.943143
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.572354
5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.091609


## Build User Profile

In [11]:
def build_user_profile(user_id):

    user_data = interactions[interactions.user_id == user_id]

    profile = np.zeros(reel_matrix.shape[1])

    for _, row in user_data.iterrows():
        reel_vector = reel_matrix.loc[row.reel_id].values
        weight = row.interaction_score
        profile += reel_vector * weight

    return profile


## Recommendation Function

In [12]:
def recommend(user_id, top_k=10):

    user_profile = build_user_profile(user_id)

    scores = cosine_similarity(
        [user_profile],
        reel_matrix
    )[0]

    result = pd.DataFrame({
        "reel_id": reel_matrix.index,
        "score": scores
    })

    result = result.sort_values(
        "score",
        ascending=False
    )

    return result.head(top_k)


## Test Recommendation

In [14]:
recommend(10,15)

,reel_id,score
2508,2509,0.812952
2661,2662,0.812824
536,537,0.812398
2420,2421,0.811450
951,952,0.811125
2840,2841,0.810469
2819,2820,0.808818
781,782,0.806554
2906,2907,0.806125
2050,2051,0.804989
